### Import Modules


In [ ]:
# Standard Library
import random
from datetime import datetime, timezone
from pathlib import Path

# Packages
import mlflow, numpy, pandas, shap
from matplotlib import pyplot
from sklearn.model_selection import train_test_split

### Constants & Config


In [ ]:
# Paths
ROOT_DIR = Path("../").resolve()

# Data
TEST_SIZE = 0.2
START_DATE = datetime(2013, 9, 1, tzinfo=timezone.utc)
SEED_PATH = (
    ROOT_DIR
    / "database"
    / "seed"
    / "transaction_inferences"
    / "creditcard_transactions.csv.gz"
)

# Reproducibility
RANDOM_STATE = 42
RNG = numpy.random.default_rng(RANDOM_STATE)
random.seed(RANDOM_STATE)
numpy.random.seed(RANDOM_STATE)

# MLflow
MLFLOW_DIR = Path("./mlflow").resolve()
MLFLOW_DIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{(MLFLOW_DIR / 'mlflow.db').as_posix()}")

### Load Data


In [ ]:
seed_df = pandas.read_csv(
    SEED_PATH,
    compression="gzip",
    dtype={
        "Time": "float64",
        **{f"V{i}": "float64" for i in range(1, 29)},
        "Amount": "float64",
        "Class": "int8",
    },
)

# Convert seconds offset to epoch seconds (int64), same as the training dataset
df = pandas.DataFrame({
    "is_fraud": seed_df["Class"].astype("int64"),
    "transaction_timestamp": (
        START_DATE.timestamp() + seed_df["Time"]
    ).astype("int64"),
    "amount": seed_df["Amount"],
    **{f"v{i}": seed_df[f"V{i}"] for i in range(1, 29)},
})

x = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

# Same split as notebook 02
train_x, test_x, train_y, test_y = train_test_split(
    x, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

test_x_np = test_x.to_numpy(dtype=numpy.float64)
test_y_np = test_y.to_numpy(dtype=numpy.int64)
feature_names = list(test_x.columns)

print(f"Data shape: {df.shape}")
print(f"Test shape: {test_x.shape}")
print(feature_names)

### Load Model


In [ ]:
xgb = mlflow.sklearn.load_model(model_uri="models:/XGBoost/latest")

print(xgb.named_steps.keys())
xgb_robust_scaler = xgb.named_steps["robust_scaler"]
xgb_model = xgb.named_steps["xgb"]
xgb_model

### Load Model Interpreter


In [ ]:
explainer = shap.TreeExplainer(xgb_model)
test_x_scaled_np = xgb_robust_scaler.transform(test_x_np)
shap_values = explainer.shap_values(test_x_scaled_np)

### Global Importance


In [ ]:
shap.summary_plot(
    shap_values,
    test_x_np,
    plot_type="bar",
    feature_names=feature_names,
    rng=RNG,
)

### Local Interpretation


In [ ]:
fraud_idx = numpy.where(test_y_np == 1)[0][0]
legit_idx = numpy.where(test_y_np == 0)[0][0]

for index, label in [
    (int(fraud_idx), "Fraud"),
    (int(legit_idx), "Legitimate"),
]:
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values[index],
            base_values=explainer.expected_value,
            data=test_x.iloc[index].to_numpy(),
            feature_names=feature_names,
        ),
        show=False,
    )
    pyplot.title(f"Local Explanation — {label} (y={test_y_np[index]})")
    pyplot.tight_layout()
    pyplot.show()